# HyperRAG-M2: End-to-End Demo

This notebook demonstrates the HyperRAG pipeline for multi-hop QA.

It walks through: corpus loading, hyperlink graph construction, FAISS indexing,
retrieval comparison (B1 / B2 / HyperRAG), and results display.

**Google Colab users:** Run Cell 2 to install dependencies first.
**Local users:** Skip Cell 2 (dependencies installed via requirements.txt).

In [ ]:
# Uncomment and run this cell in Google Colab:
# !pip install -q transformers sentence-transformers faiss-cpu networkx datasets beautifulsoup4 evaluate requests numpy torch tqdm

In [ ]:
import sys
from pathlib import Path

# Add project root to path (works from notebooks/ dir locally or Colab root)
PROJECT_ROOT = Path('..') if Path('../src').exists() else Path('.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.corpus import load_hotpotqa, build_corpus, load_corpus
from src.graph import build_and_save_graph, load_graph, print_graph_stats
from src.embeddings import build_and_save_index, load_index, search
from src.retrieval import naive_rag, htmlrag_style, hyperrag, compute_em, compute_f1

print('All imports successful!')

## 1. Load Corpus

In [ ]:
CORPUS_PATH = PROJECT_ROOT / 'data' / 'corpus.json'

if CORPUS_PATH.exists():
    print(f'Loading cached corpus from {CORPUS_PATH}')
    corpus = load_corpus(CORPUS_PATH)
    print(f'Loaded {len(corpus)} pages')
else:
    print('Corpus not found. Building from scratch (20-40 min)...')
    corpus = build_corpus(output_path=CORPUS_PATH)
    print(f'Built corpus: {len(corpus)} pages')

# Preview first page
page = corpus[0]
print(f"\nSample page: {page['title']}")
print(f"  Text length: {len(page['text'])} chars")
print(f"  Links: {page['links'][:5]}")

## 2. Build Hyperlink Graph

In [ ]:
GRAPH_PATH = PROJECT_ROOT / 'data' / 'hyperlink_graph.graphml'

graph = build_and_save_graph(corpus, output_path=GRAPH_PATH)
print_graph_stats(graph)

## 3. Build FAISS Index

In [ ]:
INDEX_DIR = PROJECT_ROOT / 'data'

index, page_ids = build_and_save_index(corpus, output_dir=INDEX_DIR)
print(f'Index: {index.ntotal} vectors, {len(page_ids)} page IDs')

## 4. Compare Retrieval Systems

We compare three retrieval systems on the same example question. B1 uses plain text, B2 preserves HTML structure, HyperRAG adds 1-hop graph expansion.

In [ ]:
DEMO_QUESTION = "What film was featured at the 2003 Cannes Film Festival and stars the actor who played Moriarty in the BBC Sherlock series?"

print(f"Question: {DEMO_QUESTION}\n")
print("=" * 60)

context_b1 = naive_rag(DEMO_QUESTION, index, corpus, k=3)
print("B1 - Naive RAG context (first 400 chars):")
print(context_b1[:400])
print()

context_b2 = htmlrag_style(DEMO_QUESTION, index, corpus, k=3)
print("B2 - HtmlRAG-style context (first 400 chars):")
print(context_b2[:400])
print()

context_hr = hyperrag(DEMO_QUESTION, index, corpus, graph, k=3, expand_k=5)
print("HyperRAG context (first 400 chars):")
print(context_hr[:400])

## 5. Evaluate on 10 Questions

In [ ]:
qa_items = load_hotpotqa(split='train', n_samples=10)

results = {}
for name, fn_kwargs in [
    ('B1_naive_rag', {'fn': naive_rag, 'extra': {}}),
    ('B2_htmlrag',   {'fn': htmlrag_style, 'extra': {}}),
    ('HyperRAG',     {'fn': hyperrag, 'extra': {'graph': graph, 'expand_k': 5}}),
]:
    fn = fn_kwargs['fn']
    extra = fn_kwargs['extra']
    em_scores, f1_scores = [], []
    for qa in qa_items:
        if name == 'HyperRAG':
            ctx = fn(qa['question'], index, corpus, k=5, **extra)
        else:
            ctx = fn(qa['question'], index, corpus, k=5)
        em_scores.append(compute_em(ctx, qa['answer']))
        f1_scores.append(compute_f1(ctx, qa['answer']))
    results[name] = {
        'avg_em': sum(em_scores) / len(em_scores),
        'avg_f1': sum(f1_scores) / len(f1_scores),
    }

print(f"{'System':<20}  {'EM':>6}  {'F1':>6}")
print("-" * 36)
for name, scores in results.items():
    print(f"{name:<20}  {scores['avg_em']:>6.3f}  {scores['avg_f1']:>6.3f}")

print("\nNote: EM/F1 here measure retrieval quality (does gold answer appear in context?)")
print("Run python run_all_systems.py for the full 50-question evaluation.")

## 6. Full Evaluation (50 Questions)

For the full 50-question evaluation used in the submission, run from the project root: `python run_all_systems.py`. Results are saved to `data/results.csv`.